# 任务 1：数据加载与概览
* 读取 CSV 文件，查看数据结构（info，describe，head）。
* 清洗数据：删除 CustomerID 为空的行，过滤 Quantity > 0 的有效购买记录。

In [ ]:
# ============================================================
# 实验3：无监督学习——聚类与关联规则
# 本代码用于Jupyter Notebook中逐任务运行，每个任务函数定义后立即调用
# 所有图表通过 plt.show() 在Jupyter中直接显示，不保存到文件
# ============================================================

# ---------- 全局导入与设置 ----------
import warnings  # warnings用于过滤不必要的警告信息
from dataclasses import dataclass  # dataclass用于快速定义简单数据类
from typing import Any, Dict, List, Tuple  # 类型标注，提高代码可读性

import matplotlib.pyplot as plt  # matplotlib.pyplot用于绘制所有图表
import numpy as np  # numpy用于数值计算
import pandas as pd  # pandas用于数据读取、清洗和分析
from mlxtend.frequent_patterns import apriori, association_rules  # Apriori与关联规则函数
from mlxtend.preprocessing import TransactionEncoder  # 事务数据One-Hot编码器
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage  # 层次聚类与树状图
from sklearn.cluster import KMeans  # K-Means聚类算法
from sklearn.decomposition import PCA  # 主成分分析，用于降维可视化
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score  # 聚类评估指标
from sklearn.preprocessing import StandardScaler  # 标准化器

try:
    import torch  # 可选：使用Torch在GPU上加速部分数值计算
    HAS_TORCH = True
except ImportError:
    torch = None
    HAS_TORCH = False

# 全局设置：过滤警告，避免Jupyter中输出过多非关键提示
warnings.filterwarnings("ignore")
# 全局设置：matplotlib中文字体，保证标题与标签能正常显示中文
plt.rcParams["font.sans-serif"] = ["SimHei"]
# 全局设置：避免负号显示成方块
plt.rcParams["axes.unicode_minus"] = False

# ---------- 全局参数 ----------
# SAMPLE_N = None 表示读取全部数据；调试时可改成 50、1000 等
SAMPLE_N = None
# 数据集绝对路径
DATA_PATH = r"data/online_retail_II.csv"
# 是否在可用时启用GPU加速（仅任务7、任务8、任务9中的部分数值计算）
USE_GPU_IF_AVAILABLE = True
# 最小支持度：全量数据下 0.01 表示至少在约 1% 的订单中出现
MIN_SUPPORT = 0.01
# 最小置信度：仅保留置信度至少为 0.5 的规则
MIN_CONFIDENCE = 0.5
# 最大频繁项集长度：限制为 3，避免Apriori组合爆炸
MAX_ITEMSET_LEN = 3


# ==================== GPU辅助函数 ====================

@dataclass
class TorchKMeansModel:
    """用于保存GPU版K-Means输出结果的简单数据类"""
    cluster_centers_: np.ndarray
    labels_: np.ndarray
    inertia_: float
    device: str


def get_gpu_status() -> Dict[str, Any]:
    status = {
        "torch_installed": HAS_TORCH,
        "cuda_available": False,
        "device": "cpu",
        "device_name": "CPU",
    }
    if HAS_TORCH and torch.cuda.is_available():
        status["cuda_available"] = True
        status["device"] = "cuda"
        status["device_name"] = torch.cuda.get_device_name(0)
    return status


def torch_kmeans_fit_predict(
    data: np.ndarray,
    n_clusters: int,
    random_state: int = 42,
    max_iter: int = 100,
    tol: float = 1e-4,
) -> TorchKMeansModel:
    if not HAS_TORCH or not torch.cuda.is_available():
        raise RuntimeError("当前环境不可用Torch CUDA。")
    device = torch.device("cuda")
    x = torch.tensor(data, dtype=torch.float32, device=device)
    n_samples = x.shape[0]
    if n_clusters >= n_samples:
        raise ValueError("聚类数必须小于样本数。")
    generator = torch.Generator(device=device)
    generator.manual_seed(random_state)
    init_idx = torch.randperm(n_samples, generator=generator, device=device)[:n_clusters]
    centers = x[init_idx].clone()
    for _ in range(max_iter):
        distances = torch.cdist(x, centers)
        labels = torch.argmin(distances, dim=1)
        new_centers = []
        for cluster_id in range(n_clusters):
            cluster_points = x[labels == cluster_id]
            if cluster_points.numel() == 0:
                new_centers.append(centers[cluster_id])
            else:
                new_centers.append(cluster_points.mean(dim=0))
        new_centers = torch.stack(new_centers)
        shift = torch.norm(new_centers - centers, dim=1).max().item()
        centers = new_centers
        if shift < tol:
            break
    final_distances = torch.cdist(x, centers)
    final_labels = torch.argmin(final_distances, dim=1)
    sq_dist = (x - centers[final_labels]) ** 2
    inertia = float(sq_dist.sum().item())
    return TorchKMeansModel(
        cluster_centers_=centers.detach().cpu().numpy(),
        labels_=final_labels.detach().cpu().numpy(),
        inertia_=inertia,
        device=str(device),
    )


# ==================== 任务1：数据加载与概览 ====================
def task1_load_and_clean(file_path: str, sample_n: int = None) -> pd.DataFrame:
    data = pd.read_csv(file_path, nrows=sample_n)
    data = data.rename(columns={
        "Customer ID": "CustomerID",
        "Price": "UnitPrice",
    })
    print("=" * 60)
    print("任务1：数据加载与概览")
    print("=" * 60)
    print("\n>>> 数据维度：")
    print(data.shape)
    print("\n>>> df.info() 输出：")
    data.info()
    print("\n>>> df.describe() 输出：")
    print(data.describe(include="all").T.head(20))
    print("\n>>> df.head() 输出：")
    print(data.head())

    cleaned = data.copy()
    cleaned["InvoiceDate"] = pd.to_datetime(cleaned["InvoiceDate"], errors="coerce")
    cleaned["Quantity"] = pd.to_numeric(cleaned["Quantity"], errors="coerce")
    cleaned["UnitPrice"] = pd.to_numeric(cleaned["UnitPrice"], errors="coerce")
    cleaned["CustomerID"] = pd.to_numeric(cleaned["CustomerID"], errors="coerce")
    cleaned["InvoiceNo"] = cleaned["Invoice"].astype(str)
    cleaned = cleaned.dropna(subset=["CustomerID"])
    cleaned = cleaned[cleaned["Quantity"] > 0]
    cleaned = cleaned[cleaned["UnitPrice"] > 0]
    cleaned = cleaned.dropna(subset=["InvoiceDate"])
    print("\n>>> 清洗后数据维度：")
    print(cleaned.shape)
    print(f"清洗前数据量：{len(data)}")
    print(f"清洗后数据量：{len(cleaned)}")
    print(f"删除无效记录数：{len(data) - len(cleaned)}")
    return cleaned

cleaned_data = task1_load_and_clean(DATA_PATH, sample_n=SAMPLE_N)

# 任务 2：构造购物篮事务数据
* 按 InvoiceNo（订单号）分组，将每笔订单中包含的 StockCode（或 Description）聚合为列表。
* 生成事务列表 transactions，格式为 [['item1','item2',...], ...]。
* 使用 mlxtend.preprocessing.TransactionEncoder 将事务转换为 One-Hot 编码的 DataFrame，便于 Apriori 算法使用。

In [ ]:
# ==================== 任务2：构造购物篮事务数据 ====================
def task2_build_transactions(cleaned: pd.DataFrame, item_col: str = "StockCode") -> Tuple[List[List[str]], pd.DataFrame]:
    print("=" * 60)
    print("任务2：构造购物篮事务数据")
    print("=" * 60)
    use_col = "StockCode" if item_col not in cleaned.columns else item_col
    temp = cleaned[["InvoiceNo", use_col]].copy()
    temp[use_col] = temp[use_col].astype(str).str.strip()
    temp = temp[temp[use_col] != ""]
    grouped = temp.groupby("InvoiceNo")[use_col].apply(lambda x: list(pd.unique(x)))
    transactions = grouped.tolist()
    encoder = TransactionEncoder()
    encoded_array = encoder.fit(transactions).transform(transactions)
    one_hot_df = pd.DataFrame(encoded_array, columns=encoder.columns_)
    print(f"订单事务数：{len(transactions)}")
    print(f"One-Hot矩阵维度：{one_hot_df.shape}")
    print("\n>>> 前3笔事务示例：")
    for i, tx in enumerate(transactions[:3]):
        print(f"订单{i + 1}: {tx}")
    return transactions, one_hot_df

transactions, one_hot_data = task2_build_transactions(cleaned_data, item_col="StockCode")

# 任务 3：寻找频繁项集
* 设定最小支持度（如 min_support=0.01，可根据数据规模调整）。
* 使用 mlxtend.frequent_patterns.apriori 找出频繁项集。
* 输出频繁项集的数量及前几项。

In [ ]:
# ==================== 任务3：寻找频繁项集 ====================
def task3_mine_frequent_itemsets(
    one_hot_df: pd.DataFrame,
    min_support: float = 0.01,
    max_len: int = 3,
) -> pd.DataFrame:
    print("=" * 60)
    print("任务3：寻找频繁项集")
    print("=" * 60)
    if one_hot_df.empty:
        print("One-Hot数据为空，无法挖掘频繁项集。")
        return pd.DataFrame()
    frequent_itemsets = apriori(
        one_hot_df,
        min_support=min_support,
        use_colnames=True,
        max_len=max_len,
        low_memory=True,
    )
    frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False).reset_index(drop=True)
    print(f"最小支持度：{min_support}")
    print(f"最大项集长度：{max_len}")
    print(f"频繁项集数量：{len(frequent_itemsets)}")
    print("\n>>> 频繁项集前10项：")
    print(frequent_itemsets.head(10))
    return frequent_itemsets

frequent_itemsets = task3_mine_frequent_itemsets(
    one_hot_data,
    min_support=MIN_SUPPORT,
    max_len=MAX_ITEMSET_LEN,
)

# 任务 4：生成关联规则
* 使用 association_rules 函数，设置最小置信度（如 min_threshold=0.5）。
* 计算每条规则的提升度（lift），筛选出提升度 > 1 的有效规则。

In [ ]:
# ==================== 任务4：生成关联规则 ====================
def task4_generate_rules(frequent_itemsets: pd.DataFrame, min_threshold: float = 0.5) -> pd.DataFrame:
    print("=" * 60)
    print("任务4：生成关联规则")
    print("=" * 60)
    if frequent_itemsets.empty or len(frequent_itemsets) < 2:
        print("频繁项集不足，无法生成关联规则。")
        return pd.DataFrame()
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=min_threshold,
    )
    rules = rules[rules["lift"] > 1].copy()
    if not rules.empty:
        rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ",".join(list(x)))
        rules["consequents_str"] = rules["consequents"].apply(lambda x: ",".join(list(x)))
        rules = rules.sort_values(by=["lift", "confidence"], ascending=False).reset_index(drop=True)
    print(f"最小置信度：{min_threshold}")
    print(f"有效规则数量（lift > 1）：{len(rules)}")
    print("\n>>> 规则前10项：")
    if not rules.empty:
        print(rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]].head(10))
    return rules

rules = task4_generate_rules(frequent_itemsets, min_threshold=MIN_CONFIDENCE)

# 任务 5：规则解读与可视化
* 选择 3~5 条具有业务意义的规则（如“购买 A 的客户也很可能购买 B”），解释支持度、置信度、提升度的含义。
* 绘制规则散点图（支持度 vs 置信度，颜色映射提升度）。

In [ ]:
# ==================== 任务5：规则解读与可视化 ====================
def task5_explain_and_visualize_rules(rules: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    print("=" * 60)
    print("任务5：规则解读与可视化")
    print("=" * 60)
    if rules.empty:
        print("无可解释规则。")
        return pd.DataFrame()
    selected_rules = rules.head(top_n).copy()
    print("\n>>> 选取的TOP规则解读：")
    for idx, row in selected_rules.iterrows():
        print(
            f"规则{idx + 1}: 购买「{row['antecedents_str']}」的客户也倾向于购买「{row['consequents_str']}」；"
            f"支持度={row['support']:.4f}；置信度={row['confidence']:.4f}；提升度={row['lift']:.4f}"
        )
    print("\n>>> 指标含义：")
    print("支持度：规则中的商品组合在全部订单中出现的比例。")
    print("置信度：购买前件A的客户中，同时购买后件B的比例。")
    print("提升度：在前件A已发生的条件下，后件B出现概率相对独立情况被放大的倍数。")

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        rules["support"],
        rules["confidence"],
        c=rules["lift"],
        cmap="viridis",
        alpha=0.8,
        edgecolors="k",
    )
    plt.colorbar(scatter, label="提升度 Lift")
    plt.xlabel("支持度 Support")
    plt.ylabel("置信度 Confidence")
    plt.title("关联规则散点图（支持度 vs 置信度，颜色映射提升度）")
    plt.tight_layout()
    plt.show()
    return selected_rules

selected_rules = task5_explain_and_visualize_rules(rules, top_n=5)

# 任务 6：构建 RFM 特征（Recency/Frequency/Monetary）
* 为每位客户（CustomerID）计算以下特征：
+ R：最近购买日期（距参考日期的天数）
+ F：购买频率（订单数）
+ M：总消费金额（Quantity * UnitPrice 之和）

In [ ]:
# ==================== 任务6：构建RFM特征 ====================
def task6_build_rfm(cleaned: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Timestamp]:
    print("=" * 60)
    print("任务6：构建RFM特征")
    print("=" * 60)
    if cleaned.empty:
        print("数据为空，无法构建RFM特征。")
        return pd.DataFrame(), pd.Timestamp.now()
    work = cleaned.copy()
    work["Amount"] = work["Quantity"] * work["UnitPrice"]
    ref_date = work["InvoiceDate"].max() + pd.Timedelta(days=1)
    rfm = (
        work.groupby("CustomerID")
        .agg(
            Recency=("InvoiceDate", lambda x: (ref_date - x.max()).days),
            Frequency=("InvoiceNo", "nunique"),
            Monetary=("Amount", "sum"),
        )
        .reset_index()
    )
    print(f"参考日期：{ref_date}")
    print(f"客户数：{len(rfm)}")
    print("\n>>> RFM前10行：")
    print(rfm.head(10))
    print("\n>>> RFM描述统计：")
    print(rfm[["Recency", "Frequency", "Monetary"]].describe())
    return rfm, ref_date

rfm_features, ref_date = task6_build_rfm(cleaned_data)

# 任务 7：数据预处理
* 处理异常值：对 RFM 特征进行对数变换（减轻偏态）。
* 标准化：使用 StandardScaler 将特征缩放到均值为 0、方差为 1。
* 使用 PCA 降维至 2 维，便于可视化。

In [ ]:
# ==================== 任务7：数据预处理 ====================
def task7_preprocess_rfm(
    rfm: pd.DataFrame,
    use_gpu: bool = False,
    gpu_status: Dict[str, Any] = None,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, StandardScaler, PCA]:
    print("=" * 60)
    print("任务7：数据预处理")
    print("=" * 60)
    if rfm.empty:
        print("RFM数据为空，无法预处理。")
        return pd.DataFrame(), np.array([]), np.array([]), StandardScaler(), PCA(n_components=2)
    rfm_features_only = rfm[["Recency", "Frequency", "Monetary"]].copy()
    print("\n>>> 原始RFM偏度：")
    print(rfm_features_only.skew().to_string())
    if use_gpu and gpu_status and gpu_status["cuda_available"]:
        device = torch.device(gpu_status["device"])
        x = torch.tensor(rfm_features_only.values, dtype=torch.float32, device=device)
        rfm_log_tensor = torch.log1p(x)
        mean = rfm_log_tensor.mean(dim=0, keepdim=True)
        std = rfm_log_tensor.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-8)
        scaled_tensor = (rfm_log_tensor - mean) / std
        rfm_log = pd.DataFrame(
            rfm_log_tensor.cpu().numpy(),
            columns=rfm_features_only.columns,
            index=rfm_features_only.index,
        )
        scaled = scaled_tensor.cpu().numpy()
        print(f"\n已使用GPU进行对数变换和标准化：{gpu_status['device_name']}")
    else:
        rfm_log = np.log1p(rfm_features_only)
        scaler_tmp = StandardScaler()
        scaled = scaler_tmp.fit_transform(rfm_log)
        print("\n当前使用CPU进行对数变换和标准化。")
    print("\n>>> 对数变换后RFM偏度：")
    print(rfm_log.skew().to_string())
    scaler = StandardScaler()
    scaler.fit(rfm_log)
    pca = PCA(n_components=2, random_state=42)
    pca_data = pca.fit_transform(scaled)
    print("\n>>> 标准化后均值：")
    print(scaled.mean(axis=0).round(6))
    print("\n>>> 标准化后标准差：")
    print(scaled.std(axis=0).round(6))
    print("\n>>> PCA解释方差比：")
    print(pca.explained_variance_ratio_)
    print(f"累计解释方差比例：{pca.explained_variance_ratio_.sum()}")
    return rfm_log, scaled, pca_data, scaler, pca

gpu_status = get_gpu_status()
rfm_log, scaled_data, pca_data, scaler, pca_model = task7_preprocess_rfm(
    rfm_features,
    use_gpu=USE_GPU_IF_AVAILABLE,
    gpu_status=gpu_status,
)

# 任务 8：确定最佳聚类数
* 计算不同 K 值（2~10）的 轮廓系数（silhouette score）和 簇内误差平方和（inertia）。
* 绘制肘部图，选择拐点对应的 K 值。

In [ ]:
# ==================== 任务8：确定最佳聚类数 ====================
def task8_select_optimal_k(
    scaled_data: np.ndarray,
    min_k: int = 2,
    max_k: int = 10,
    use_gpu: bool = False,
    gpu_status: Dict[str, Any] = None,
) -> Tuple[pd.DataFrame, int]:
    print("=" * 60)
    print("任务8：确定最佳聚类数")
    print("=" * 60)
    if scaled_data.size == 0:
        print("标准化数据为空，无法评估K值。")
        return pd.DataFrame(), 2
    n_samples = scaled_data.shape[0]
    use_max_k = max(min_k, min(max_k, n_samples - 1))
    k_list = []
    silhouette_list = []
    inertia_list = []
    using_gpu_kmeans = use_gpu and gpu_status and gpu_status["cuda_available"]
    for k in range(min_k, use_max_k + 1):
        if using_gpu_kmeans:
            model = torch_kmeans_fit_predict(scaled_data, n_clusters=k, random_state=42)
            labels = model.labels_
            inertia = model.inertia_
        else:
            model = KMeans(n_clusters=k, random_state=42, n_init=20)
            labels = model.fit_predict(scaled_data)
            inertia = model.inertia_
        silhouette = silhouette_score(scaled_data, labels) if len(set(labels)) > 1 else np.nan
        k_list.append(k)
        silhouette_list.append(silhouette)
        inertia_list.append(inertia)
    metrics_df = pd.DataFrame({
        "k": k_list,
        "silhouette": silhouette_list,
        "inertia": inertia_list,
    })
    best_row = metrics_df.sort_values(by="silhouette", ascending=False).iloc[0]
    best_k = int(best_row["k"])
    print("\n>>> 各K值评估结果：")
    print(metrics_df.to_string(index=False))
    print(f"\n建议最佳K值：{best_k}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(metrics_df["k"], metrics_df["inertia"], marker="o")
    axes[0].set_xlabel("K值")
    axes[0].set_ylabel("簇内误差平方和 Inertia")
    axes[0].set_title("肘部图")
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(metrics_df["k"], metrics_df["silhouette"], marker="o", color="orange")
    axes[1].set_xlabel("K值")
    axes[1].set_ylabel("轮廓系数 Silhouette")
    axes[1].set_title("轮廓系数曲线")
    axes[1].grid(True, alpha=0.3)
    axes[1].annotate(
        f"最佳K={best_k}",
        xy=(best_k, best_row["silhouette"]),
        xytext=(best_k + 0.5, best_row["silhouette"] - 0.02),
        arrowprops=dict(arrowstyle="->", color="red"),
        fontsize=11,
        color="red",
    )
    plt.tight_layout()
    plt.show()
    return metrics_df, best_k

k_metrics_df, optimal_k = task8_select_optimal_k(
    scaled_data,
    min_k=2,
    max_k=10,
    use_gpu=USE_GPU_IF_AVAILABLE,
    gpu_status=gpu_status,
)

# 任务 9：执行 K-Means 聚类
* 使用选定的 K 值训练 K-Means 模型。
* 将聚类标签添加到原始客户特征表中。

In [ ]:
# ==================== 任务9：执行K-Means聚类 ====================
def task9_kmeans_clustering(
    scaled_data: np.ndarray,
    rfm: pd.DataFrame,
    best_k: int,
    use_gpu: bool = False,
    gpu_status: Dict[str, Any] = None,
) -> Tuple[pd.DataFrame, Any, np.ndarray]:
    print("=" * 60)
    print("任务9：执行K-Means聚类")
    print("=" * 60)
    if scaled_data.size == 0 or rfm.empty:
        print("数据为空，无法执行K-Means聚类。")
        return pd.DataFrame(), KMeans(n_clusters=2, random_state=42), np.array([])
    using_gpu_kmeans = use_gpu and gpu_status and gpu_status["cuda_available"]
    if using_gpu_kmeans:
        model = torch_kmeans_fit_predict(scaled_data, n_clusters=best_k, random_state=42)
        labels = model.labels_
        device_name = gpu_status["device_name"]
    else:
        model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
        labels = model.fit_predict(scaled_data)
        device_name = "CPU"
    result = rfm.copy()
    result["KMeansCluster"] = labels
    print(f"K-Means执行设备：{device_name}")
    print(f"聚类数 K = {best_k}")
    print("\n>>> 各簇样本数量：")
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    for cluster_id, count in cluster_counts.items():
        print(f"簇{cluster_id}：{count}位客户")
    print("\n>>> 带聚类标签的前10行：")
    print(result.head(10))
    return result, model, labels

kmeans_result, kmeans_model, kmeans_labels = task9_kmeans_clustering(
    scaled_data,
    rfm_features,
    optimal_k,
    use_gpu=USE_GPU_IF_AVAILABLE,
    gpu_status=gpu_status,
)

# 任务 10：结果可视化与业务解读
* 使用散点图（可结合 PCA 降维后的二维坐标）展示不同簇的分布。
* 统计每个簇的中心点（原始特征尺度下的均值），分析各簇客户特征（如高价值簇、低频簇等）。
* 将聚类结果与关联规则挖掘得到的商品组合结合，提出针对不同簇的营销建议。

In [ ]:
# ==================== 任务10：结果可视化与业务解读 ====================
def task10_visualize_and_interpret(
    pca_data: np.ndarray,
    kmeans_result: pd.DataFrame,
    selected_rules: pd.DataFrame,
    optimal_k: int,
) -> pd.DataFrame:
    print("=" * 60)
    print("任务10：结果可视化与业务解读")
    print("=" * 60)
    if pca_data.size == 0 or kmeans_result.empty:
        print("数据为空，无法进行可视化与解读。")
        return pd.DataFrame()
    labels = kmeans_result["KMeansCluster"].values
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        pca_data[:, 0],
        pca_data[:, 1],
        c=labels,
        cmap="tab10",
        alpha=0.8,
        edgecolors="k",
    )
    plt.colorbar(scatter, label="簇标签 Cluster")
    plt.xlabel("PCA 第一主成分")
    plt.ylabel("PCA 第二主成分")
    plt.title(f"K-Means聚类结果（PCA降维至2维，K={optimal_k}）")
    plt.tight_layout()
    plt.show()

    profile = (
        kmeans_result.groupby("KMeansCluster")[["Recency", "Frequency", "Monetary"]]
        .mean()
        .reset_index()
        .sort_values(by="Monetary", ascending=False)
    )
    print("\n>>> 各簇RFM均值：")
    print(profile.to_string(index=False))
    print("\n>>> 营销建议：")
    median_monetary = profile["Monetary"].median()
    median_frequency = profile["Frequency"].median()
    median_recency = profile["Recency"].median()
    for _, row in profile.iterrows():
        cid = int(row["KMeansCluster"])
        if row["Monetary"] >= median_monetary and row["Frequency"] >= median_frequency:
            print(f"簇{cid}：高价值高频客户，建议推送会员权益与高端组合套餐。")
        elif row["Recency"] > median_recency:
            print(f"簇{cid}：沉睡风险客户，建议用召回券与限时促销进行激活。")
        else:
            print(f"簇{cid}：潜力成长客户，建议用满减、加价购提升客单与频率。")
    if not selected_rules.empty:
        sample_rule = selected_rules.iloc[0]
        print("\n>>> 联动陈列建议：")
        print(
            f"可基于规则「{sample_rule['antecedents_str']}」→「{sample_rule['consequents_str']}」"
            f"（置信度={sample_rule['confidence']:.2f}, 提升度={sample_rule['lift']:.2f}）做推荐。"
        )
    return profile

profile = task10_visualize_and_interpret(
    pca_data,
    kmeans_result,
    selected_rules,
    optimal_k=optimal_k,
)

# 任务 11：层次聚类与树状图
* 使用 scipy.cluster.hierarchy 的 linkage 方法（方法可选 ward，average 等）对标准化后的数据进行层次聚类。
* 绘制树状图（dendrogram），并选择合适的距离阈值切割出 K 个簇。

In [ ]:
# ==================== 任务11：层次聚类与树状图 ====================
def task11_hierarchical_clustering(
    scaled_data: np.ndarray,
    target_k: int = 3,
    method: str = "ward",
) -> Tuple[np.ndarray, np.ndarray]:
    print("=" * 60)
    print("任务11：层次聚类与树状图")
    print("=" * 60)
    if scaled_data.size == 0:
        print("数据为空，无法进行层次聚类。")
        return np.array([]), np.array([])
    z_matrix = linkage(scaled_data, method=method)
    labels = fcluster(z_matrix, t=target_k, criterion="maxclust") - 1
    print(f"链接方法：{method}")
    print(f"目标簇数：{target_k}")
    print("\n>>> 各簇样本数量：")
    cluster_distribution = pd.Series(labels).value_counts().sort_index()
    for cluster_id, count in cluster_distribution.items():
        print(f"簇{cluster_id}：{count}位客户")
    plt.figure(figsize=(12, 5))
    dendrogram(
        z_matrix,
        truncate_mode="lastp",
        p=20,
        leaf_rotation=45,
        leaf_font_size=10,
        show_contracted=True,
    )
    if target_k > 1 and len(z_matrix) >= target_k - 1:
        cut_idx = max(1, target_k - 1)
        cut_height = z_matrix[-cut_idx, 2]
        plt.axhline(
            y=cut_height,
            color="red",
            linestyle="--",
            alpha=0.7,
            label=f"切割阈值（K={target_k}）",
        )
        plt.legend()
    plt.title(f"层次聚类树状图（链接方法：{method}，截断显示）")
    plt.xlabel("样本簇（截断后）")
    plt.ylabel("合并距离")
    plt.tight_layout()
    plt.show()
    return labels, z_matrix

hierarchical_labels, z_matrix = task11_hierarchical_clustering(
    scaled_data,
    target_k=optimal_k,
    method="ward",
)

# 任务 12：聚类结果对比
* 将层次聚类得到的簇标签与 K-Means 的结果进行比较（可使用调整兰德指数或互信息，若真实标签未知则仅定性对比）。
* 分析两种算法在客户分组上的异同，讨论各自的适用场景。

In [ ]:
# ==================== 任务12：聚类结果对比 ====================
def task12_compare_cluster_results(
    kmeans_labels: np.ndarray,
    hierarchical_labels: np.ndarray,
    scaled_data: np.ndarray,
) -> Dict[str, float]:
    print("=" * 60)
    print("任务12：聚类结果对比")
    print("=" * 60)
    result = {
        "ARI": np.nan,
        "NMI": np.nan,
        "Silhouette_KMeans": np.nan,
        "Silhouette_Hierarchical": np.nan,
    }
    if len(kmeans_labels) == 0 or len(hierarchical_labels) == 0:
        print("聚类标签为空，无法比较。")
        return result
    result["ARI"] = adjusted_rand_score(kmeans_labels, hierarchical_labels)
    result["NMI"] = normalized_mutual_info_score(kmeans_labels, hierarchical_labels)
    print(f"调整兰德指数（ARI）：{result['ARI']:.4f}")
    print(f"归一化互信息（NMI）：{result['NMI']:.4f}")
    print("\n>>> 算法对比分析：")
    print("K-Means：速度快，适合大规模数据，适合发现近似球状簇。")
    print("层次聚类：结果直观，可通过树状图观察层次结构，但复杂度更高。")
    return result

comparison_result = task12_compare_cluster_results(
    kmeans_labels,
    hierarchical_labels,
    scaled_data,
)

# 任务 13：聚类评估（内部指标）
* 计算两种聚类的轮廓系数，对比其聚类紧密度与分离度。

In [ ]:
# ==================== 任务13：聚类评估（内部指标） ====================
def task13_evaluate_clusters(
    kmeans_labels: np.ndarray,
    hierarchical_labels: np.ndarray,
    scaled_data: np.ndarray,
    comparison_result: Dict[str, float],
) -> None:
    print("=" * 60)
    print("任务13：聚类评估（内部指标）")
    print("=" * 60)
    if len(kmeans_labels) == 0 or len(hierarchical_labels) == 0:
        print("聚类标签为空，无法评估。")
        return None
    sil_kmeans = silhouette_score(scaled_data, kmeans_labels) if len(set(kmeans_labels)) > 1 else np.nan
    sil_hier = silhouette_score(scaled_data, hierarchical_labels) if len(set(hierarchical_labels)) > 1 else np.nan
    comparison_result["Silhouette_KMeans"] = sil_kmeans
    comparison_result["Silhouette_Hierarchical"] = sil_hier
    print(f"K-Means轮廓系数：{sil_kmeans:.4f}")
    print(f"层次聚类轮廓系数：{sil_hier:.4f}")
    if not np.isnan(sil_kmeans) and not np.isnan(sil_hier):
        diff = sil_kmeans - sil_hier
        if diff > 0.05:
            print(f"结论：K-Means聚类效果更好（高出 {diff:.4f}）。")
        elif diff < -0.05:
            print(f"结论：层次聚类效果更好（高出 {-diff:.4f}）。")
        else:
            print(f"结论：两种聚类效果接近（差异 {diff:.4f}）。")
    return None

task13_evaluate_clusters(
    kmeans_labels,
    hierarchical_labels,
    scaled_data,
    comparison_result,
)

# ==================== 全部任务完成 ====================
print("=" * 60)
print("全部分析任务完成！")
print("=" * 60)